# Week 11 · Day 5 — English → Urdu Translation (Seq2Seq)

The capstone of the sequence-models week: an **Encoder–Decoder** that reads an English sentence and writes the Urdu translation. This is where the two GRUs have **different jobs** — one *understands*, one *generates*.

```
 English sentence  →  Encoder GRU  →  context state  →  Decoder GRU  →  Urdu sentence
```

**Today:**
1. The big idea: encoder–decoder
2. Prepare the English–Urdu data
3. Build the seq2seq model (with **teacher forcing**)
4. Train and **translate** new sentences
5. **Swap the embedding**: learned → **Word2Vec** → **GloVe**, and compare

> **Kaggle GPU:** Settings → Accelerator → GPU. Add the **english-corpus.txt** and **urdu-corpus.txt** files as inputs.
> **Internet ON** for the Word2Vec/GloVe downloads in Part 5.

---
## 1. The big idea: Encoder–Decoder

Translation is **many-to-many**, but input and output lengths differ ("I am a student" = 4 words → "میں ایک طالب علم ہوں" = 5 words). We use **two** GRUs:

- **Encoder GRU** — reads the whole English sentence and squeezes it into a single **context vector** (its final hidden state). Its job: *understand*.
- **Decoder GRU** — starts from that context and generates the Urdu translation **one word at a time**. Its job: *generate*.

```
 "I" → "am" → "a" → "student"          (encoder reads English)
                        ↓
                  context state
                        ↓
 <sos> → میں → طالب → علم → ہوں → <eos>   (decoder writes Urdu)
```

The **`<sos>`** (start) and **`<eos>`** (end) tokens tell the decoder when to begin and stop.

> 🖼️ **IMAGE NEEDED** — search prompt: **"encoder decoder seq2seq GRU machine translation architecture diagram"**  
> *(The standard encoder–decoder diagram: encoder RNN → context vector → decoder RNN producing output words. Central visual for the whole day.)*

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.layers import Input, Embedding, GRU, Dense
from tensorflow.keras.models import Model

tf.random.set_seed(42)
print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

---
## 2. Prepare the data

We have a **parallel corpus**: two files with aligned lines — line *n* of the English file translates to line *n* of the Urdu file.

In [ ]:
# set these to your Kaggle input paths (check the file browser on the right)
EN_PATH = "english-corpus.txt"   # e.g. /kaggle/input/eng-urdu/english-corpus.txt
UR_PATH = "urdu-corpus.txt"

en_lines = open(EN_PATH, encoding="utf-8").read().strip().split("\n")
ur_lines = open(UR_PATH, encoding="utf-8").read().strip().split("\n")

# keep only non-empty, aligned pairs
pairs = [(e.strip(), u.strip()) for e, u in zip(en_lines, ur_lines) if e.strip() and u.strip()]
print("total pairs:", len(pairs))
for e, u in pairs[:5]:
    print(f"  {e:30s} -> {u}")

In [ ]:
# to train fast in class, use a subset of the short sentences
MAX_PAIRS = 8000
pairs = [(e, u) for e, u in pairs if len(e.split()) <= 6 and len(u.split()) <= 8][:MAX_PAIRS]

en_texts = [e for e, u in pairs]
# wrap every Urdu target with start/end tokens so the decoder knows where to begin & stop
ur_texts = ["<sos> " + u + " <eos>" for e, u in pairs]
print("using", len(pairs), "pairs")
print("example target:", ur_texts[0])

### Tokenize both languages
Each language gets its **own** tokenizer (separate vocabularies). Note `filters=''` for Urdu so the `<sos>`/`<eos>` tokens aren't stripped.

In [ ]:
en_tok = Tokenizer()
en_tok.fit_on_texts(en_texts)
en_vocab = len(en_tok.word_index) + 1

ur_tok = Tokenizer(filters='')          # keep <sos> and <eos>
ur_tok.fit_on_texts(ur_texts)
ur_vocab = len(ur_tok.word_index) + 1

print("English vocab:", en_vocab, "| Urdu vocab:", ur_vocab)

In [ ]:
# convert to padded integer sequences
en_seq = en_tok.texts_to_sequences(en_texts)
ur_seq = ur_tok.texts_to_sequences(ur_texts)

MAX_EN = max(len(s) for s in en_seq)
MAX_UR = max(len(s) for s in ur_seq)
en_seq = pad_sequences(en_seq, maxlen=MAX_EN, padding="post")
ur_seq = pad_sequences(ur_seq, maxlen=MAX_UR, padding="post")
print("English padded:", en_seq.shape, "| Urdu padded:", ur_seq.shape)

### Teacher forcing: set up decoder input & target

**Teacher forcing** means: during training we feed the decoder the **correct previous Urdu word** (not its own guess), so it learns fast and stably.

- **decoder input** = the Urdu sequence **without the last word** (starts with `<sos>`)
- **decoder target** = the Urdu sequence **shifted one step left** (what it should predict next)

```
 input : <sos>  میں   طالب  علم   ہوں
 target:  میں   طالب  علم   ہوں   <eos>
```

In [ ]:
decoder_input  = ur_seq[:, :-1]   # everything except the last token
decoder_target = ur_seq[:, 1:]    # shifted left by one
print("decoder input :", decoder_input.shape)
print("decoder target:", decoder_target.shape)

---
## 3. Build the seq2seq model (learned embeddings)

We build it as one training model with two inputs (English sentence + decoder input). A reusable function lets us **swap the embedding layer** later (Part 5) without rewriting everything.

In [ ]:
LATENT = 256      # size of the GRU hidden state (the 'memory')
EMB_DIM = 100     # embedding dimension

def build_seq2seq(en_embedding_layer, ur_embedding_layer):
    """Build encoder-decoder. Pass in the embedding layers so we can swap them."""
    # --- Encoder ---
    enc_inputs = Input(shape=(MAX_EN,), name="encoder_input")
    enc_emb = en_embedding_layer(enc_inputs)
    _, enc_state = GRU(LATENT, return_state=True, name="encoder_gru")(enc_emb)

    # --- Decoder ---
    dec_inputs = Input(shape=(MAX_UR - 1,), name="decoder_input")
    dec_emb = ur_embedding_layer(dec_inputs)
    dec_gru = GRU(LATENT, return_sequences=True, return_state=True, name="decoder_gru")
    dec_seq, _ = dec_gru(dec_emb, initial_state=enc_state)   # start decoder from encoder's context
    dec_dense = Dense(ur_vocab, activation="softmax", name="output")
    outputs = dec_dense(dec_seq)

    model = Model([enc_inputs, dec_inputs], outputs)
    # keep references to the pieces we need for inference
    model.enc_inputs, model.enc_state = enc_inputs, enc_state
    model.dec_gru, model.dec_dense, model.ur_embedding = dec_gru, dec_dense, ur_embedding_layer
    return model

In [ ]:
# version 1: BOTH embeddings learned from scratch during training
en_emb_learned = Embedding(en_vocab, EMB_DIM, name="en_emb")
ur_emb_learned = Embedding(ur_vocab, EMB_DIM, name="ur_emb")

model = build_seq2seq(en_emb_learned, ur_emb_learned)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
history = model.fit(
    [en_seq, decoder_input], decoder_target[..., None],
    epochs=30, batch_size=128, validation_split=0.1, verbose=1)

---
## 4. Translate new sentences (inference)

At inference we **don't have the correct Urdu words** to feed in — so we generate one word at a time, feeding each prediction back in. We build two small models from the trained pieces:
- an **encoder model**: English → context state,
- a **step-decoder**: (previous word + state) → (next word + new state).

In [ ]:
def make_inference(model):
    # encoder: input sentence -> context state
    encoder = Model(model.enc_inputs, model.enc_state)

    # step decoder: one word + previous state -> next-word probs + new state
    step_word = Input(shape=(1,))
    state_in = Input(shape=(LATENT,))
    emb = model.ur_embedding(step_word)
    seq, state_out = model.dec_gru(emb, initial_state=state_in)
    probs = model.dec_dense(seq)
    step_decoder = Model([step_word, state_in], [probs, state_out])
    return encoder, step_decoder

encoder_model, step_decoder = make_inference(model)
idx_to_word = {i: w for w, i in ur_tok.word_index.items()}
print("inference models ready")

In [ ]:
def translate(sentence, encoder, decoder):
    seq = pad_sequences(en_tok.texts_to_sequences([sentence.lower()]), maxlen=MAX_EN, padding="post")
    state = encoder.predict(seq, verbose=0)
    word = np.array([[ur_tok.word_index["<sos>"]]])
    output = []
    for _ in range(MAX_UR):
        probs, state = decoder.predict([word, state], verbose=0)
        idx = int(probs[0, -1].argmax())
        w = idx_to_word.get(idx, "")
        if w == "<eos>" or w == "":
            break
        output.append(w)
        word = np.array([[idx]])
    return " ".join(output)

# try it on a few sentences
for s in ["i am happy", "how are you", "what is your name", "i am a student"]:
    print(f"{s:22s} -> {translate(s, encoder_model, step_decoder)}")

The translations won't be perfect on a small dataset trained briefly — but you should see it produce real Urdu words in a sensible order. **You built a translator.**

---
## 5. Swap the embedding: learned → Word2Vec → GloVe

So far the **English** embeddings were learned from scratch on our small dataset. But we can instead start from **pretrained** embeddings that already know word meanings from huge corpora — the same **transfer-learning** idea as vision.

We'll try two pretrained sources for the **English** side:
- **GloVe** (Stanford) and **Word2Vec** (Google) — both give a vector per English word.

*(Urdu stays learned-from-scratch: good pretrained Urdu vectors are less standard, so this cleanly isolates the effect of pretrained English embeddings.)*

In [ ]:
import gensim.downloader as api

# download pretrained vectors (needs internet; cached after first time)
print("downloading GloVe (100-dim)...")
glove = api.load("glove-wiki-gigaword-100")     # 100-dim to match EMB_DIM
print("downloading Word2Vec (300-dim)...")
w2v = api.load("word2vec-google-news-300")
print("done. glove dim:", glove.vector_size, "| w2v dim:", w2v.vector_size)

### Build an embedding matrix from pretrained vectors
For each English word in **our** vocabulary, look up its pretrained vector and place it in a matrix. Words not found keep a zero row. We then load this matrix into a (frozen) `Embedding` layer.

In [ ]:
def make_pretrained_embedding(kv, dim):
    """Build a Keras Embedding for the ENGLISH vocab from pretrained vectors kv."""
    matrix = np.zeros((en_vocab, dim))
    found = 0
    for word, i in en_tok.word_index.items():
        if word in kv:
            matrix[i] = kv[word]
            found += 1
    print(f"  matched {found}/{en_vocab-1} English words in the pretrained vectors")
    return Embedding(en_vocab, dim, weights=[matrix], trainable=False, name="en_emb_pretrained")

In [ ]:
def train_variant(en_embedding, dim, label):
    """Build + train a seq2seq with the given ENGLISH embedding; return final val accuracy."""
    print(f"\n=== {label} ===")
    ur_emb = Embedding(ur_vocab, dim, name="ur_emb")   # urdu always learned, matching dim
    m = build_seq2seq(en_embedding, ur_emb)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = m.fit([en_seq, decoder_input], decoder_target[..., None],
              epochs=30, batch_size=128, validation_split=0.1, verbose=0)
    acc = h.history["val_accuracy"][-1]
    print(f"{label}: validation accuracy {acc:.2%}")
    return acc, m

results = {}

# 1. learned (re-use dim 100 for a fair compare)
acc_learned, _ = train_variant(Embedding(en_vocab, 100, name="en_emb"), 100, "Learned (from scratch)")
results["Learned"] = acc_learned

# 2. GloVe (100-dim)
acc_glove, _ = train_variant(make_pretrained_embedding(glove, 100), 100, "GloVe (pretrained)")
results["GloVe"] = acc_glove

# 3. Word2Vec (300-dim)
acc_w2v, _ = train_variant(make_pretrained_embedding(w2v, 300), 300, "Word2Vec (pretrained)")
results["Word2Vec"] = acc_w2v

In [ ]:
import matplotlib.pyplot as plt

names = list(results.keys())
accs = [results[n] * 100 for n in names]
plt.bar(names, accs, color=["gray", "steelblue", "green"])
plt.ylabel("validation accuracy (%)")
plt.title("English embedding source: learned vs GloVe vs Word2Vec")
for i, v in enumerate(accs):
    plt.text(i, v + 0.3, f"{v:.1f}", ha="center")
plt.show()

print("summary:")
for n in names:
    print(f"  {n:10s} {results[n]:.2%}")

**What to look for:**
- Pretrained embeddings (**GloVe/Word2Vec**) often help most when the training data is **small** — they bring in word meaning our tiny corpus can't teach.
- On a very small/clean dataset the learned embeddings can catch up, since the model can memorize. The benefit of pretrained vectors grows as vocabulary and rarity increase.
- This is the **same transfer-learning lesson** as vision: *reuse* knowledge from a big corpus instead of learning everything from scratch. *(Exact numbers vary per run.)*

---
## Your turn (solo task) ✍️

Pick at least two:
1. **Translate your own sentences** with the trained model — which ones work, which fail, and why?
2. **Train longer** (60 epochs) or on **more pairs** (raise `MAX_PAIRS`) — do translations improve?
3. **Compare embeddings fairly** — make GloVe and the learned version both 100-dim (already done) and read the gap.
4. **Make the decoder deeper** or raise `LATENT` — effect on translation quality?
5. **Think ahead:** the encoder crams the *whole* sentence into one context vector. What happens for a *long* sentence? (This is the problem **attention** solves — next.)

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====



---
## Summary

- **Seq2seq translation** uses **two GRUs**: an **encoder** that compresses the English sentence into a context vector, and a **decoder** that generates Urdu word-by-word from it.
- **`<sos>`/`<eos>`** tokens mark where the decoder starts and stops.
- **Teacher forcing** feeds the correct previous word during training for fast, stable learning; at **inference** we feed the model's own predictions back, one word at a time.
- **Embeddings:** we trained with **learned**, then **GloVe**, then **Word2Vec** English embeddings — pretrained vectors are transfer learning for language, most helpful when data is limited.

### The limitation to remember
The encoder squeezes the **entire** sentence into **one** vector. For long sentences, that's a bottleneck — the context vector "forgets" the start.

> **That bottleneck is exactly what *attention* fixes** — letting the decoder look back at every input word. That's where we go next: **attention, then Transformers.**

**This completes Week 11's sequence-models arc:** RNN → LSTM/GRU → embeddings → seq2seq translation → (next) attention.